In [0]:
from datetime import datetime

 

from pyspark.sql import functions as F

from pyspark.sql.window import Window

from delta.tables import DeltaTable

 

SCHEMA = "automotive_catalog.automotive_mfg"

SILVER_VEHICLE = f"{SCHEMA}.silver_vehicle_production"

SILVER_QUALITY = f"{SCHEMA}.silver_quality_inspection"

SILVER_SUPPLIER = f"{SCHEMA}.silver_supplier_delivery"

GOLD_TABLE = f"{SCHEMA}.gold_vehicle_manufacturing_kpi"

CONTROL_ROW = "gold_vehicle_manufacturing_kpi"

 

EPOCH = datetime(1900, 1, 1)

 

def audit_log(message):

    print(f"[AUDIT] {message}")

 

audit_log("Starting Gold KPI Processing")

In [0]:
# ---------------------------------------------------------

# 1) Read this table's own watermark (silver processed_ts)

# ---------------------------------------------------------

wm_row = spark.sql(f"""

    SELECT last_processed_ts

    FROM {SCHEMA}.control_metadata

    WHERE table_name = '{CONTROL_ROW}'

""").collect()

last_wm = wm_row[0]["last_processed_ts"] if wm_row else None

if last_wm is None:

    last_wm = EPOCH

In [0]:
inc_vehicle = spark.table(SILVER_VEHICLE).filter(F.col("processed_ts") > F.lit(last_wm))

inc_quality = spark.table(SILVER_QUALITY).filter(F.col("processed_ts") > F.lit(last_wm))

inc_supplier = spark.table(SILVER_SUPPLIER).filter(F.col("processed_ts") > F.lit(last_wm))

 

inc_counts = {

    "vehicle": inc_vehicle.count(),

    "quality": inc_quality.count(),

    "supplier": inc_supplier.count(),

}

 

if sum(inc_counts.values()) == 0:

    audit_log(f"No new silver rows past watermark {last_wm} - nothing to process")

else:

    # Watermark to advance to: max processed_ts actually consumed

    new_wm = (

        inc_vehicle.select(F.col("processed_ts").alias("ts"))

        .unionByName(inc_quality.select(F.col("processed_ts").alias("ts")))

        .unionByName(inc_supplier.select(F.col("processed_ts").alias("ts")))

        .agg(F.max("ts"))

        .collect()[0][0]

    )

In [0]:
changed_plants = inc_supplier.select("plant_code").distinct()

 

# Supplier KPIs are plant level -> every vehicle at a changed

# plant is affected too.

plants_vins = (

    spark.table(SILVER_VEHICLE)

    .join(changed_plants, "plant_code")

    .select("vin_number")

)



affected_vins = (

    inc_vehicle.select("vin_number")

    .unionByName(inc_quality.select("vin_number"))

    .unionByName(plants_vins)

    .distinct()

    .persist()

)

In [0]:
vehicle_df = (

        spark.table(SILVER_VEHICLE)

        .join(affected_vins, "vin_number")

        .withColumn(

            "rn",

            F.row_number().over(

                Window.partitionBy("vin_number").orderBy(F.col("processed_ts").desc())

            ),

        )

        .filter(F.col("rn") == 1)

        .drop("rn")

    )



# Plants of the affected VINs may also need fresh supplier KPIs

affected_plants = (

    vehicle_df.select("plant_code")

    .unionByName(changed_plants)

    .distinct()

)

In [0]:
quality_agg = (

        spark.table(SILVER_QUALITY)

        .join(affected_vins, "vin_number")

        .groupBy("vin_number")

        .agg(

            F.count("*").alias("total_defects"),

            F.sum(

                F.when(F.col("defect_severity").isin("HIGH", "CRITICAL"), 1).otherwise(0)

            ).alias("high_severity_defects"),

            F.max("rework_required").alias("rework_required_flag"),

        )

    )

In [0]:
supplier_agg = (

        spark.table(SILVER_SUPPLIER)

        .join(affected_plants, "plant_code")

        .groupBy("plant_code")

        .agg(

            F.sum("planned_qty").alias("planned_parts_qty"),

            F.sum("delivered_qty").alias("delivered_parts_qty"),

            F.max(

                F.when(F.col("delivery_status") == "DELAYED", "Y").otherwise("N")

            ).alias("supplier_delay_flag"),

        )

        .withColumn(

            "delivery_fill_rate",

            F.round((F.col("delivered_parts_qty") / F.col("planned_parts_qty")) * 100, 2),

        )

    )

In [0]:

# ---------------------------------------------------------

# 5) Stage: one row per affected VIN

# ---------------------------------------------------------
gold_stage = (
    vehicle_df.alias("v")
    .join(quality_agg.alias("q"), "vin_number", "left")
    .join(
        supplier_agg.alias("s"),
        F.col("v.plant_code") == F.col("s.plant_code"),
        "left",
    )
    .select(
        F.col("v.vin_number"),
        F.col("v.plant_code"),
        F.col("v.assembly_line"),
        F.col("v.model_name"),
        F.col("v.production_stage"),
        F.col("v.production_status"),
        F.coalesce(F.col("q.total_defects"), F.lit(0)).alias("total_defects"),
        F.coalesce(F.col("q.high_severity_defects"), F.lit(0)).alias("high_severity_defects"),
        F.coalesce(F.col("q.rework_required_flag"), F.lit("N")).alias("rework_required_flag"),
        F.coalesce(F.col("s.supplier_delay_flag"), F.lit("N")).alias("supplier_delay_flag"),
        F.coalesce(F.col("s.planned_parts_qty"), F.lit(0)).alias("planned_parts_qty"),
        F.coalesce(F.col("s.delivered_parts_qty"), F.lit(0)).alias("delivered_parts_qty"),
        F.coalesce(F.col("s.delivery_fill_rate"), F.lit(0.0)).alias("delivery_fill_rate"),
        F.col("v.op_code"),
    )
    .withColumn(
        "quality_risk_level",
        F.when(F.col("high_severity_defects") >= 2, "HIGH")
        .when(F.col("high_severity_defects") == 1, "MEDIUM")
        .otherwise("LOW"),
    )
)

# ---------------------------------------------------------
# 6) SCD2 step 1: expire current rows whose tracked attributes
#    changed (the old version stays as history)
# ---------------------------------------------------------
gold_dt = DeltaTable.forName(spark, GOLD_TABLE)

(
    gold_dt.alias("tgt")
    .merge(
        gold_stage.alias("src"),
        "tgt.vin_number = src.vin_number AND tgt.is_current = true",
    )
    .whenMatchedUpdate(
        condition="""
            tgt.production_stage <> src.production_stage
            OR tgt.production_status <> src.production_status
            OR tgt.total_defects <> src.total_defects
            OR tgt.high_severity_defects <> src.high_severity_defects
            OR tgt.quality_risk_level <> src.quality_risk_level
            OR tgt.supplier_delay_flag <> src.supplier_delay_flag
            OR tgt.delivery_fill_rate <> src.delivery_fill_rate
        """,
        set={
            "is_current": "false",
            "effective_to": "current_timestamp()",
            "updated_ts": "current_timestamp()",
        },
    )
    .execute()
)

# ---------------------------------------------------------
# 7) SCD2 step 2: insert fresh current rows for new / changed
#    VINs (VINs whose current row survived step 1 are skipped
#    by the anti-join, so nothing is duplicated)
# ---------------------------------------------------------
current_vins = spark.sql(
    f"SELECT vin_number FROM {GOLD_TABLE} WHERE is_current = true"
)

(
    gold_stage.join(current_vins, "vin_number", "left_anti")
    .select(
        F.col("vin_number"),
        F.col("plant_code"),
        F.col("assembly_line"),
        F.col("model_name"),
        F.col("production_stage"),
        F.col("production_status"),
        F.col("total_defects"),
        F.col("high_severity_defects"),
        F.col("rework_required_flag"),
        F.col("supplier_delay_flag"),
        F.col("planned_parts_qty"),
        F.col("delivered_parts_qty"),
        F.col("delivery_fill_rate"),
        F.col("quality_risk_level"),
        F.col("op_code"),
        F.lit(True).alias("is_current"),
        F.current_timestamp().alias("effective_from"),
        F.lit("9999-12-31").cast("timestamp").alias("effective_to"),
        F.current_timestamp().alias("created_ts"),
        F.current_timestamp().alias("updated_ts"),
    )
    .write.format("delta")
    .mode("append")
    .saveAsTable(GOLD_TABLE)
)

# ---------------------------------------------------------
# 8) Advance own watermark only after a successful write
# ---------------------------------------------------------
spark.sql(f"""
    UPDATE {SCHEMA}.control_metadata
    SET last_processed_ts = '{new_wm}', updated_at = current_timestamp()
    WHERE table_name = '{CONTROL_ROW}'
""")

affected_vins.unpersist()

audit_log(f"Increment {inc_counts} -> new watermark {new_wm}")

audit_log("Completed Gold KPI Processing")

In [0]:
%sql

select * from automotive_catalog.automotive_mfg.gold_vehicle_manufacturing_kpi